# 🚲 Post-Deploy Setup — Ontology, Graph Model & Operations Agent

Run this notebook **after** `Deploy_Bicycle_RTI.ipynb` completes.

It creates items that fabric-cicd doesn't support natively:
1. **Bicycle_Ontology_Model_New** (Ontology) — 12 entity types, 23 relationships
2. **Bicycle_Ontology_Model_New_graph** (GraphModel) — connected to bicycles_gold lakehouse
3. **Cycling-Campaign-Agent** (OperationsAgent) — campaign automation agent

### Prerequisites
- `Deploy_Bicycle_RTI.ipynb` has completed successfully
- Pipeline `PL_BicycleRTI_Medallion` has run at least once (tables must exist)

In [ ]:
# =============================================================
# CELL 1 — Setup & Discovery
# =============================================================
import json, requests, time, base64, os
import sempy.fabric as fabric
from notebookutils import mssparkutils

# Get workspace info
ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# Get auth token
token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Discover bicycles_gold lakehouse ID
items = fabric.list_items()
gold_lh = items[items['Display Name'] == 'bicycles_gold']
if gold_lh.empty:
    raise RuntimeError("bicycles_gold lakehouse not found! Run Deploy_Bicycle_RTI.ipynb first.")
gold_lh_id = gold_lh.iloc[0]['Id']
print(f"bicycles_gold lakehouse: {gold_lh_id}")

# Discover Bicycle Ontology Model SM ID
ont_sm = items[items['Display Name'] == 'Bicycle Ontology Model']
ont_sm_id = ont_sm.iloc[0]['Id'] if not ont_sm.empty else None
print(f"Bicycle Ontology Model SM: {ont_sm_id}")

print("\n✅ Discovery complete")

In [ ]:
# =============================================================
# CELL 2 — Deploy Ontology (Bicycle_Ontology_Model_New)
# =============================================================
# Uses the Fabric Ontology REST API to:
#  1. Create the ontology item
#  2. Load the full definition (12 entity types, 23 relationships,
#     data bindings, contextualizations) from disk
#  3. Patch workspace/lakehouse IDs for the target environment
#  4. Push the definition via updateDefinition API
#
# API: POST /v1/workspaces/{id}/ontologies
# API: POST /v1/workspaces/{id}/ontologies/{id}/updateDefinition
# Ref: https://learn.microsoft.com/en-us/rest/api/fabric/ontology/items

import re

ONTOLOGY_NAME = "Bicycle_Ontology_Model_New"
ONT_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/ontologies"

def load_ontology_parts(base_path):
    """Walk the ontology directory and build base64-encoded parts."""
    parts = []
    for root, _dirs, files in sorted(os.walk(base_path)):
        for fname in sorted(files):
            if fname == ".platform":
                continue
            filepath = os.path.join(root, fname)
            rel_path = os.path.relpath(filepath, base_path).replace("\\", "/")
            with open(filepath, 'rb') as f:
                raw = f.read()
            if raw.startswith(b'\xef\xbb\xbf'):
                raw = raw[3:]
            payload = base64.b64encode(raw).decode("utf-8")
            parts.append({"path": rel_path, "payload": payload, "payloadType": "InlineBase64"})
    return parts

def patch_bindings(parts, target_ws_id, target_lh_id):
    """Patch workspace/lakehouse IDs in DataBinding and Contextualization parts."""
    patched = []
    for part in parts:
        path = part["path"]
        if "DataBindings/" in path or "Contextualizations/" in path:
            try:
                content = base64.b64decode(part["payload"]).decode("utf-8")
                obj = json.loads(content)
                _patch_ids(obj, target_ws_id, target_lh_id)
                content = json.dumps(obj, indent=2, ensure_ascii=False)
                part = {**part, "payload": base64.b64encode(content.encode("utf-8")).decode("utf-8")}
            except Exception as e:
                print(f"    [WARN] Could not patch {path}: {e}")
        patched.append(part)
    return patched

def _patch_ids(obj, ws_id, lh_id):
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            val = obj[key]
            lk = key.lower()
            if lk in ("workspaceid", "workspaceguid", "workspace_id"):
                obj[key] = ws_id
            elif lk in ("itemid", "lakehouseid", "artifactid", "item_id"):
                obj[key] = lh_id
            elif isinstance(val, str) and "onelake" in val.lower():
                m = re.match(r'(abfss://)([0-9a-f-]+)(@onelake[^/]*/)([0-9a-f-]+)(.*)', val, re.I)
                if m:
                    obj[key] = f"{m.group(1)}{ws_id}{m.group(3)}{lh_id}{m.group(5)}"
            elif isinstance(val, (dict, list)):
                _patch_ids(val, ws_id, lh_id)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, (dict, list)):
                _patch_ids(item, ws_id, lh_id)

def wait_lro(response, label, timeout=180):
    loc = response.headers.get("Location")
    if not loc:
        time.sleep(10)
        return True
    start = time.time()
    retry = int(response.headers.get("Retry-After", 5))
    while time.time() - start < timeout:
        time.sleep(retry)
        r = requests.get(loc, headers=headers)
        if r.status_code == 200:
            status = r.json().get("status", "")
            if status == "Succeeded":
                return True
            if status in ("Failed", "Cancelled"):
                print(f"    [FAIL] {label}: {status}")
                return False
    return False

print(f"Deploying Ontology: {ONTOLOGY_NAME}")
print(f"  Workspace: {ws_id}")
print(f"  Lakehouse: {gold_lh_id}")

# Check if already exists
existing_ont = items[items['Display Name'] == ONTOLOGY_NAME]
if not existing_ont.empty:
    ont_id = existing_ont.iloc[0]['Id']
    print(f"  Already exists (ID: {ont_id}) — will update definition")
else:
    # Create empty ontology via dedicated API
    create_body = {
        "displayName": ONTOLOGY_NAME,
        "description": "Bicycle Fleet Ontology — 12 entity types, 23 relationships, bound to bicycles_gold",
    }
    r = requests.post(ONT_API, headers=headers, json=create_body)
    if r.status_code in (200, 201):
        ont_id = r.json().get("id", "")
        print(f"  Created: {ont_id}")
    elif r.status_code == 202:
        wait_lro(r, "create ontology")
        time.sleep(3)
        # Find by name
        r2 = requests.get(ONT_API, headers=headers)
        ont_id = None
        for o in r2.json().get("value", []):
            if o["displayName"] == ONTOLOGY_NAME:
                ont_id = o["id"]
                break
        print(f"  Created (async): {ont_id}")
    else:
        print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")
        ont_id = None

if ont_id:
    # Load definition from disk
    ont_dir = "/lakehouse/default/Files/RTI-Hackathon-Demo/post_deploy/definitions/Bicycle_Ontology_Model_New.Ontology"
    if not os.path.exists(ont_dir):
        print(f"  [WARN] Ontology dir not found at {ont_dir}")
        print(f"         Upload the repo to lakehouse Files first")
    else:
        parts = load_ontology_parts(ont_dir)
        et_count = sum(1 for p in parts if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"))
        rt_count = sum(1 for p in parts if p["path"].startswith("RelationshipTypes/") and p["path"].endswith("/definition.json"))
        db_count = sum(1 for p in parts if "DataBindings/" in p["path"])
        ctx_count = sum(1 for p in parts if "Contextualizations/" in p["path"])
        print(f"  Loaded {len(parts)} parts: {et_count} entities, {rt_count} relationships, {db_count} bindings, {ctx_count} contextualizations")

        # Patch IDs for target environment
        parts = patch_bindings(parts, ws_id, gold_lh_id)
        print(f"  Patched data bindings for target environment")

        # Push full definition
        update_url = f"{ONT_API}/{ont_id}/updateDefinition"
        body = {"definition": {"parts": parts}}
        r = requests.post(update_url, headers=headers, json=body)
        if r.status_code in (200, 201):
            print(f"  ✅ Definition pushed successfully")
        elif r.status_code == 202:
            ok = wait_lro(r, "updateDefinition")
            print(f"  ✅ Definition pushed successfully" if ok else "  [FAIL] updateDefinition failed")
        else:
            print(f"  [FAIL] updateDefinition: {r.status_code} {r.text[:300]}")

        # Verify
        verify_url = f"{ONT_API}/{ont_id}/getDefinition"
        vr = requests.post(verify_url, headers=headers)
        if vr.status_code == 200:
            vparts = vr.json().get("definition", {}).get("parts", [])
            print(f"  Verified: {len(vparts)} parts in ontology definition")
        elif vr.status_code == 202:
            print(f"  Verification pending (LRO) — check in Fabric UI")

print("\n✅ Ontology step complete")

In [ ]:
# =============================================================
# CELL 3 — Deploy Graph Model
# =============================================================
# The graph model is auto-provisioned when you open the ontology in
# the Fabric UI, but we can also deploy a standalone graph model via
# the GraphModel REST API for full programmatic control.
#
# API: POST /v1/workspaces/{id}/graphModels
# API: POST /v1/workspaces/{id}/graphModels/{id}/updateDefinition
# API: POST /v1/workspaces/{id}/graphModels/{id}/jobs/refreshGraph/instances
# Ref: https://learn.microsoft.com/en-us/rest/api/fabric/graphmodel/items
#
# NOTE: Graph refresh for ontology-managed graphs returns InvalidJobType.
#       Only the Fabric UI 'Refresh now' button works for auto-provisioned
#       graphs. Standalone graphs CAN be refreshed via API.

GRAPH_MODEL_NAME = "Bicycle_Ontology_Model_New_graph"
GM_API = f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/graphModels"

print(f"Deploying GraphModel: {GRAPH_MODEL_NAME}")

# Check if already exists
existing_gm = items[items['Display Name'].str.contains(GRAPH_MODEL_NAME, na=False)]
if not existing_gm.empty:
    gm_id = existing_gm.iloc[0]['Id']
    print(f"  Already exists (ID: {gm_id}) — will update definition")
else:
    gm_id = None

# Load definition from disk
gm_dir = "/lakehouse/default/Files/RTI-Hackathon-Demo/post_deploy/definitions/Bicycle_Ontology_Model_New_graph.GraphModel"
gm_files = ['graphType.json', 'dataSources.json', 'graphDefinition.json', 'stylingConfiguration.json']

gm_parts = []
if os.path.exists(gm_dir):
    for gm_file in gm_files:
        filepath = os.path.join(gm_dir, gm_file)
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                content = f.read()
            # Patch workspace/lakehouse placeholders
            content = content.replace("__WORKSPACE_ID__", ws_id)
            content = content.replace("__LH_BICYCLES_GOLD__", gold_lh_id)
            encoded = base64.b64encode(content.encode('utf-8')).decode('utf-8')
            gm_parts.append({"path": gm_file, "payload": encoded, "payloadType": "InlineBase64"})
            print(f"  Loaded: {gm_file}")
        else:
            print(f"  [WARN] Missing: {gm_file}")
else:
    print(f"  [WARN] Graph model dir not found at {gm_dir}")
    print(f"         Upload the repo to lakehouse Files first")

if gm_parts:
    if gm_id:
        # Update existing definition
        update_url = f"{GM_API}/{gm_id}/updateDefinition"
        body = {"definition": {"parts": gm_parts}}
        r = requests.post(update_url, headers=headers, json=body)
        if r.status_code in (200, 201):
            print(f"  ✅ Definition updated")
        elif r.status_code == 202:
            ok = wait_lro(r, "updateDefinition")
            print(f"  ✅ Definition updated" if ok else "  [FAIL] Update failed")
        else:
            print(f"  [FAIL] updateDefinition: {r.status_code} {r.text[:300]}")
    else:
        # Create new graph model with definition
        create_body = {
            "displayName": GRAPH_MODEL_NAME,
            "description": "Graph model for Bicycle Fleet Ontology — 12 node types, 23 edge types",
            "definition": {"parts": gm_parts},
        }
        r = requests.post(GM_API, headers=headers, json=create_body)
        if r.status_code in (200, 201):
            gm_id = r.json().get('id', '')
            print(f"  ✅ Created: {gm_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create graph model")
            if ok:
                time.sleep(3)
                r2 = requests.get(GM_API, headers=headers)
                for g in r2.json().get("value", []):
                    if GRAPH_MODEL_NAME in g.get("displayName", ""):
                        gm_id = g["id"]
                        break
                print(f"  ✅ Created (async): {gm_id}")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")

    # Try refresh (will fail for ontology-managed graphs)
    if gm_id:
        print(f"\n  Attempting graph refresh...")
        refresh_url = f"{GM_API}/{gm_id}/jobs/refreshGraph/instances"
        r = requests.post(refresh_url, headers=headers)
        if r.status_code in (200, 202):
            print(f"  Refresh triggered — check Fabric UI for status")
        else:
            error = r.json().get("error", {}) if r.text else {}
            code = error.get("errorCode", "")
            if "InvalidJobType" in code or "InvalidJobType" in r.text:
                print(f"  [INFO] InvalidJobType — this is an ontology-managed graph")
                print(f"         Use Fabric UI > Graph Model > 'Refresh now' button")
            else:
                print(f"  [WARN] Refresh: {r.status_code} — may need manual refresh")
else:
    print("  No definition parts loaded — skipping graph creation")

print("\n✅ Graph model step complete")

In [ ]:
# =============================================================
# CELL 4 — Deploy Operations Agent (Cycling-Campaign-Agent)
# =============================================================
# Uses the generic items API since OperationsAgent doesn't have
# a dedicated REST API endpoint yet.

print("Creating OperationsAgent: Cycling-Campaign-Agent ...")

existing_oa = items[items['Display Name'] == 'Cycling-Campaign-Agent']
if not existing_oa.empty:
    print(f"  Already exists (ID: {existing_oa.iloc[0]['Id']}) — skipping")
else:
    oa_dir = "/lakehouse/default/Files/RTI-Hackathon-Demo/post_deploy/definitions/Cycling-Campaign-Agent.OperationsAgent"
    oa_files = ['Configurations.json']
    
    parts = []
    if os.path.exists(oa_dir):
        for oa_file in oa_files:
            filepath = os.path.join(oa_dir, oa_file)
            if os.path.exists(filepath):
                with open(filepath, 'r') as f:
                    content = f.read()
                encoded = base64.b64encode(content.encode('utf-8')).decode('utf-8')
                parts.append({"path": oa_file, "payload": encoded, "payloadType": "InlineBase64"})
    else:
        print(f"  [WARN] Agent dir not found at {oa_dir}")

    if parts:
        create_body = {
            "displayName": "Cycling-Campaign-Agent",
            "type": "OperationsAgent",
            "definition": {"parts": parts},
        }
        r = requests.post(
            f"https://api.fabric.microsoft.com/v1/workspaces/{ws_id}/items",
            headers=headers,
            json=create_body
        )
        if r.status_code in (200, 201):
            oa_id = r.json().get('id', '')
            print(f"  ✅ Created: {oa_id}")
        elif r.status_code == 202:
            ok = wait_lro(r, "create operations agent")
            print(f"  ✅ Created (async)" if ok else "  [FAIL] Create failed")
        else:
            print(f"  [FAIL] Create: {r.status_code} {r.text[:300]}")
    else:
        print("  No definition files — skipping")

print("\n✅ Operations agent step complete")

## ✅ Post-Deploy Complete!

### What was deployed

| Item | API | Status |
|------|-----|--------|
| **Bicycle_Ontology_Model_New** | `POST /ontologies` + `updateDefinition` | 12 entity types, 23 relationships, data bindings |
| **Bicycle_Ontology_Model_New_graph** | `POST /graphModels` + `updateDefinition` | 4-part definition (graphType, dataSources, graphDefinition, styling) |
| **Cycling-Campaign-Agent** | `POST /items` | Operations agent with campaign instructions |

### Remaining Manual Steps

1. **Refresh Graph Model**: Open `Bicycle_Ontology_Model_New_graph` in Fabric UI → click **Refresh now**
   - API refresh returns `InvalidJobType` for ontology-managed graphs
   - This is a known limitation — only the UI button works
2. **Run the Pipeline**: Open `PL_BicycleRTI_Medallion` → Run (first load ~15-25 min)
3. **Start Eventstreams**: Open `RTIbikeRental` and `RTI-WeatherDemo` → verify they are streaming
4. **Test Data Agent**: Open `Bicycle Fleet Intelligence Agent` → ask: *"What are the top 5 busiest stations?"*